# 04 — Attention Deep Dive: What Do Attention Heads See?

This notebook dissects the attention mechanism in **Qwen 2.5 0.5B** head by head.

By the end, you'll know:
- How to read an attention pattern matrix (who attends to whom)
- What different head "types" look like (previous-token, BOS, semantic)
- How GQA groups share keys/values while diverging on queries
- Which specific heads capture high-level semantics like coreference

**Requirements**: ~2GB RAM. No GPU needed.

In [ ]:
import sys
sys.path.insert(0, "..")
import torch
import numpy as np
import matplotlib.pyplot as plt
from utils.model_loading import load_tlens_model
from utils.visualization import (
    plot_attention_pattern, plot_attention_grid,
    apply_theme, ACCENT_BLUE, ACCENT_ORANGE, ACCENT_GREEN, ACCENT_RED,
    ACCENT_PURPLE, ACCENT_TEAL, TEXT_COLOR, DARK_BG, DARK_SURFACE, DARK_GRID,
    PALETTE, _style_box,
)
apply_theme()

MODEL_SIZE = "0.5b"
model = load_tlens_model(MODEL_SIZE)

prompt = "The cat sat on the mat because it was tired"
with torch.no_grad():
    logits, cache = model.run_with_cache(prompt)
tokens = model.to_str_tokens(prompt)
print(f"Tokens: {tokens}")
print(f"Sequence length: {len(tokens)}")

## Single Head: Reading an Attention Pattern

An attention pattern is a matrix where cell `(i, j)` shows how much the token at position `i` **attends to** the token at position `j`.

Key things to notice:
- **Causal mask**: the upper triangle is always zero — tokens can only attend to previous tokens (and themselves). This is what makes the model autoregressive.
- **Rows sum to 1**: each row is a probability distribution (output of softmax) over all previous positions.
- **Bright cells** = high attention weight. A bright cell at `(i, j)` means "when predicting what comes after position `i`, the model is pulling information from position `j`."

In [ ]:
# Layer 0, Head 0 — often a simple positional pattern
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

pattern_L0H0 = cache["blocks.0.attn.hook_pattern"][0, 0]
plot_attention_pattern(pattern_L0H0, tokens, title="Layer 0, Head 0", ax=ax1)

# Layer 0, Head 3 — likely a different pattern
pattern_L0H3 = cache["blocks.0.attn.hook_pattern"][0, 3]
plot_attention_pattern(pattern_L0H3, tokens, title="Layer 0, Head 3", ax=ax2)

plt.tight_layout()
plt.show()

print("Compare the two heads: they attend to completely different positions.")
print("Each head learns its own 'attention strategy' — this is why multi-head attention is powerful.")

## All Heads in a Layer

Let's see every head in a single layer at once. This makes it easy to spot which heads specialize in what.

In [ ]:
n_heads = model.cfg.n_heads
n_layers = model.cfg.n_layers

# All heads in layer 0 (early layer — mostly positional patterns)
patterns_L0 = cache["blocks.0.attn.hook_pattern"][0]  # [n_heads, seq, seq]
fig = plot_attention_grid(patterns_L0, tokens, n_heads, title_prefix="L0 H")
fig.suptitle("Layer 0: All Attention Heads (early layer)", fontsize=14, y=1.02)
plt.show()

# All heads in the last layer (late layer — more semantic patterns)
last_layer = n_layers - 1
patterns_last = cache[f"blocks.{last_layer}.attn.hook_pattern"][0]
fig = plot_attention_grid(patterns_last, tokens, n_heads, title_prefix=f"L{last_layer} H")
fig.suptitle(f"Layer {last_layer}: All Attention Heads (final layer)", fontsize=14, y=1.02)
plt.show()

print(f"Early layers tend toward simple positional patterns (diagonal, column-0).")
print(f"Later layers show more complex, context-dependent patterns.")

## Head Pattern Taxonomy

Attention heads tend to fall into recognizable categories:

| Pattern Type | What It Looks Like | What It Does |
|---|---|---|
| **Previous-token** | Bright diagonal one step below the main diagonal | Copies information from the immediately preceding token — essential for bigram statistics |
| **BOS / First-token** | Bright first column | Attends to the beginning-of-sequence token — often a "default" when there's nothing more relevant |
| **Induction** | Copies a pattern from earlier in the sequence | Enables in-context learning by finding "if A followed B before, and A appears again, predict B" |
| **Positional** | Banded or striped pattern | Attends to tokens at fixed relative positions (e.g., 2 back, 3 back) |
| **Semantic** | Scattered bright cells linking related tokens | Captures meaning-based relationships: coreference, subject-verb agreement, etc. |

Let's classify heads automatically using simple heuristics.

In [ ]:
# Classify every head by its dominant attention pattern
seq_len = len(tokens)
prev_token_scores = np.zeros((n_layers, n_heads))
bos_scores = np.zeros((n_layers, n_heads))
entropy_scores = np.zeros((n_layers, n_heads))

for layer in range(n_layers):
    pattern = cache[f"blocks.{layer}.attn.hook_pattern"][0]  # [n_heads, seq, seq]
    for head in range(n_heads):
        attn = pattern[head].detach().float().cpu().numpy()

        # Previous-token score: mean attention on the -1 diagonal (position i attends to i-1)
        diag_vals = np.array([attn[i, i - 1] for i in range(1, seq_len)])
        prev_token_scores[layer, head] = diag_vals.mean()

        # BOS score: mean attention on column 0 (excluding row 0 which must attend to itself)
        bos_scores[layer, head] = attn[1:, 0].mean()

        # Entropy: how diffuse is the attention? High = spread out, low = focused
        # Only compute over the causal (lower-triangular) portion
        eps = 1e-10
        row_entropies = []
        for i in range(1, seq_len):
            row = attn[i, :i + 1]
            h = -np.sum(row * np.log(row + eps))
            row_entropies.append(h)
        entropy_scores[layer, head] = np.mean(row_entropies)

# Classify: dominant pattern wins
max_entropy = np.log(seq_len)  # theoretical max
labels = np.empty((n_layers, n_heads), dtype=object)
for layer in range(n_layers):
    for head in range(n_heads):
        prev = prev_token_scores[layer, head]
        bos = bos_scores[layer, head]
        if prev > 0.4:
            labels[layer, head] = "prev-token"
        elif bos > 0.4:
            labels[layer, head] = "BOS"
        elif entropy_scores[layer, head] > 0.7 * max_entropy:
            labels[layer, head] = "diffuse"
        else:
            labels[layer, head] = "other"

# Plot heatmaps
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

im0 = axes[0].imshow(prev_token_scores, cmap="magma", aspect="auto", vmin=0, vmax=1)
axes[0].set_title("Previous-Token Score", color=TEXT_COLOR)
axes[0].set_xlabel("Head")
axes[0].set_ylabel("Layer")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(bos_scores, cmap="magma", aspect="auto", vmin=0, vmax=1)
axes[1].set_title("BOS (First-Token) Score", color=TEXT_COLOR)
axes[1].set_xlabel("Head")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(entropy_scores, cmap="magma", aspect="auto", vmin=0, vmax=max_entropy)
axes[2].set_title("Attention Entropy", color=TEXT_COLOR)
axes[2].set_xlabel("Head")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.show()

# Summary counts
from collections import Counter
counts = Counter(labels.flatten())
print("Head classification summary:")
for label, count in counts.most_common():
    print(f"  {label:<15} {count:>3} heads ({count / (n_layers * n_heads) * 100:.1f}%)")

## GQA in Action: Shared Keys and Values

Qwen 2.5 0.5B uses **Grouped Query Attention** — multiple query heads share the same key and value projections. This means heads within a GQA group attend to the same "address space" (same K and V), but each query head asks a *different question* (different Q projection).

The result: heads in the same group produce **related but distinct** attention patterns. They agree on roughly *where* to look, but weight the positions differently.

In [ ]:
# GQA group structure
n_kv_heads = model.cfg.n_key_value_heads
gqa_ratio = n_heads // n_kv_heads
print(f"Query heads:   {n_heads}")
print(f"KV heads:      {n_kv_heads}")
print(f"GQA ratio:     {gqa_ratio}:1 (every {gqa_ratio} query heads share 1 KV head)")
print(f"KV groups:     {n_kv_heads}")
print()

# Show the first KV group: query heads 0 through (gqa_ratio - 1) share KV head 0
layer_idx = 0
group_heads = list(range(gqa_ratio))
print(f"First KV group (layer {layer_idx}): query heads {group_heads} share KV head 0")

patterns_layer = cache[f"blocks.{layer_idx}.attn.hook_pattern"][0]  # [n_heads, seq, seq]
group_patterns = patterns_layer[group_heads]  # [gqa_ratio, seq, seq]

fig = plot_attention_grid(group_patterns, tokens, gqa_ratio,
                          title_prefix=f"L{layer_idx} H", cols=min(gqa_ratio, 4))
fig.suptitle(f"GQA Group 0 (Layer {layer_idx}): {gqa_ratio} Query Heads Sharing KV Head 0",
             fontsize=13, y=1.02)
plt.show()

# Quantify similarity within group vs across groups
group0_patterns = patterns_layer[:gqa_ratio].reshape(gqa_ratio, -1).float()
group1_patterns = patterns_layer[gqa_ratio:2*gqa_ratio].reshape(gqa_ratio, -1).float()

# Cosine similarity within group 0
norms0 = group0_patterns / group0_patterns.norm(dim=1, keepdim=True)
within_sim = (norms0 @ norms0.T).cpu().numpy()
within_avg = (within_sim.sum() - np.trace(within_sim)) / (gqa_ratio * (gqa_ratio - 1))

# Cosine similarity between group 0 and group 1
norms1 = group1_patterns / group1_patterns.norm(dim=1, keepdim=True)
across_sim = (norms0 @ norms1.T).cpu().numpy()
across_avg = across_sim.mean()

print(f"\nAvg cosine similarity within GQA group 0:  {within_avg:.3f}")
print(f"Avg cosine similarity across groups 0 vs 1: {across_avg:.3f}")
print(f"Heads in the same group are more similar — they share the same KV projections.")

## Finding Semantic Heads

The prompt "The cat sat on the mat because **it** was tired" contains a coreference: **"it"** refers to **"cat"**. A model that understands this should have at least one attention head where the token "it" attends strongly to "cat".

Let's find which heads capture this relationship.

In [ ]:
# Find positions of "it" and "cat" in the token list
print(f"Tokens: {list(enumerate(tokens))}")

# Identify the positions (these may vary slightly by tokenizer)
it_positions = [i for i, t in enumerate(tokens) if t.strip().lower() == "it"]
cat_positions = [i for i, t in enumerate(tokens) if t.strip().lower() == "cat"]
print(f'"it" at positions:  {it_positions}')
print(f'"cat" at positions: {cat_positions}')

# Use the first occurrence of each
it_pos = it_positions[0]
cat_pos = cat_positions[0]

# For every head in every layer, get the attention weight from "it" -> "cat"
it_to_cat = np.zeros((n_layers, n_heads))
for layer in range(n_layers):
    pattern = cache[f"blocks.{layer}.attn.hook_pattern"][0]  # [n_heads, seq, seq]
    for head in range(n_heads):
        it_to_cat[layer, head] = pattern[head, it_pos, cat_pos].item()

# Find top 5 heads
flat_indices = np.argsort(it_to_cat.flatten())[::-1][:5]
print(f'\nTop 5 heads for "it" -> "cat" attention:')
print(f'{"Rank":<6} {"Layer":<8} {"Head":<8} {"Attention Weight":<18}')
print("-" * 40)
for rank, idx in enumerate(flat_indices):
    layer = idx // n_heads
    head = idx % n_heads
    weight = it_to_cat[layer, head]
    print(f"{rank + 1:<6} L{layer:<7} H{head:<7} {weight:.4f}")

# Plot the winning head's full attention pattern
best_idx = flat_indices[0]
best_layer = best_idx // n_heads
best_head = best_idx % n_heads
best_pattern = cache[f"blocks.{best_layer}.attn.hook_pattern"][0, best_head]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Heatmap of it->cat scores across all heads
im = ax1.imshow(it_to_cat, cmap="magma", aspect="auto", vmin=0)
ax1.set_title('"it" -> "cat" Attention Weight by Head', color=TEXT_COLOR)
ax1.set_xlabel("Head")
ax1.set_ylabel("Layer")
plt.colorbar(im, ax=ax1, fraction=0.046)
# Mark the best head
ax1.plot(best_head, best_layer, "s", color=ACCENT_TEAL, markersize=10, markeredgecolor=TEXT_COLOR)

# Full pattern of the best head
plot_attention_pattern(best_pattern, tokens,
                       title=f"Best Coreference Head: L{best_layer} H{best_head}",
                       ax=ax2)

plt.tight_layout()
plt.show()

print(f'\nThe winning head (L{best_layer} H{best_head}) puts {it_to_cat[best_layer, best_head]:.1%} '
      f'of "it"\'s attention on "cat".')
print("This is a semantic head — it has learned to resolve pronoun coreference.")

## Summary

**What we found:**

1. **Attention heads specialize.** Even in a small 0.5B model, different heads learn distinct roles: some track the previous token (local context), some default to BOS (a learned "no-op"), and others capture semantic relationships.

2. **GQA shares computation efficiently.** Heads within the same KV group produce correlated but non-identical attention patterns. The shared keys and values define a common "address space," while separate query projections let each head ask its own question. This cuts KV-cache memory by the GQA ratio with minimal quality loss.

3. **Some heads capture high-level semantics.** We found specific heads where "it" attends strongly to "cat" — evidence that the model learns pronoun coreference resolution as an emergent property of next-token prediction, without any explicit supervision for this task.

**Next**: [06_forward_pass_trace.ipynb](./06_forward_pass_trace.ipynb) traces actual activations through the full pipeline, and [07_logit_lens.ipynb](./07_logit_lens.ipynb) decodes the residual stream at every layer to see predictions evolve.